# DOD Particle Lab — Analysis

Loads every collected run under `experiments/data/<layout>/*/runs.csv`, joins the
hardware sidecar (`hardware.json`) on `machine_id`, and produces the per-layout
champion grid + the performance landscapes. **No global winner** (§16.9):
every champion declaration carries regime + numbers + blueprint.

Re-run after dropping in runs from new machines:
```
.venv/bin/jupyter nbconvert --execute --to notebook --inplace experiments/results/analyze.ipynb
```
(or open in JupyterLab and Run All).

In [ ]:
import glob, json, os, re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

DATA_GLOB = '../data/*/*/runs.csv'   # relative to experiments/results/
HW_GLOB   = '../data/*/*/hardware.json'

mpl.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})

## 1. Load runs + hardware

In [ ]:
# When executed by nbconvert from the repo root, cwd is the repo root and
# the relative glob resolves under experiments/results/. Handle both.
if os.path.isdir('experiments/data'):
    DATA_GLOB = 'experiments/data/*/*/runs.csv'
    HW_GLOB   = 'experiments/data/*/*/hardware.json'
else:
    DATA_GLOB = '../data/*/*/runs.csv'
    HW_GLOB   = '../data/*/*/hardware.json'

rows = []
for p in sorted(glob.glob(DATA_GLOB)):
    df = pd.read_csv(p)
    df['source_file'] = p
    rows.append(df)
runs = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

hw_rows = []
for p in sorted(glob.glob(HW_GLOB)):
    with open(p) as f: hw_rows.append(json.load(f))
hw = pd.DataFrame(hw_rows).drop_duplicates('machine_id') if hw_rows else pd.DataFrame()

n_run_files = len(rows)
n_rows = len(runs)
n_machines = runs['machine_id'].nunique() if n_rows else 0
n_cells = runs['cell'].nunique() if n_rows else 0
print(f'loaded {n_rows} rows from {n_run_files} run dirs '
      f'across {n_machines} machine(s), {n_cells} cell(s)')
if n_rows:
    print('machines:', ', '.join(sorted(runs['machine_id'].unique())))
    print('cells   :', ', '.join(sorted(runs['cell'].unique())))

## 2. Tidy + reduce (min per (cell, mode, N) across trials)

In [ ]:
if n_rows:
    # numeric coercion (blank render_ns/step_ns -> NaN)
    for c in ['ns_frame','ns_particle','gbs_eff','step_ns','render_ns']:
        runs[c] = pd.to_numeric(runs[c], errors='coerce')
    runs['death_q'] = pd.to_numeric(runs['death_q'])
    # cell short label (drop the L1. prefix for plot legibility)
    runs['cell_short'] = runs['cell'].str.replace(r'^L\d+\.', '', regex=True)

# Min-of-trials: the cleanest sample per (machine, cell, mode, death_q, N).
if n_rows:
    key = ['machine_id','cell','mode','death_q','N']
    mins = (runs.sort_values('ns_frame')
             .groupby(key, as_index=False).first()
             [['machine_id','cell','cell_short','mode','death_q','N',
               'bytes_per_particle','ns_frame','ns_particle','gbs_eff',
               'step_ns','render_ns']])
else:
    mins = pd.DataFrame()
print(f'{len(mins)} reduced rows (min-of-trials)')

## 3. Performance landscapes — ns/particle vs N

One curve per cell, per mode, per machine. Lower is better. The shape tells
the regime: flat = bandwidth-bound (ns/particle ≈ bytes ÷ DRAM BW); rising
at small N = overhead/latency-bound; rising at large N = cache spill.

In [ ]:
if len(mins):
    for (mid, mode), g in mins.groupby(['machine_id','mode']):
        fig, ax = plt.subplots(figsize=(7,4.5))
        for cell, cg in g.groupby('cell_short', sort=False):
            cg = cg.sort_values('N')
            ax.plot(cg['N'], cg['ns_particle'], 'o-', label=cell, ms=4)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('N (particles)'); ax.set_ylabel('ns / particle')
        ax.set_title(f'{mode} mode — {mid}')
        ax.legend(fontsize=8, loc='best')
        fig.tight_layout(); plt.show()
else:
    print('no data — run scripts/collect.sh first')

## 4. Effective bandwidth vs the DRAM ceiling

`gbs_eff` (step mode only) compared to the machine's single-core streaming
ceiling (from `hardware.json`). Near the ceiling = bandwidth-bound (layout
wins are byte-reduction); well below = compute/latency-bound.

In [ ]:
if len(mins) and len(hw) and not mins.query("mode == 'step'").empty:
    step = mins.query("mode == 'step'").copy()
    # join memsize for context (not a true BW ceiling, but a sanity anchor)
    step = step.merge(hw[['machine_id','cpu','memsize_bytes']], on='machine_id')
    for mid, g in step.groupby('machine_id'):
        fig, ax = plt.subplots(figsize=(7,4.5))
        for cell, cg in g.groupby('cell_short', sort=False):
            cg = cg.sort_values('N')
            ax.plot(cg['N'], cg['gbs_eff'], 'o-', label=cell, ms=4)
        ax.set_xscale('log')
        ax.set_xlabel('N'); ax.set_ylabel('GB/s eff (hot-loop bandwidth)')
        ax.set_title(f'step bandwidth — {mid} ({g["cpu"].iloc[0]})')
        ax.legend(fontsize=8, loc='best')
        fig.tight_layout(); plt.show()
else:
    print('no step-mode data — run scripts/collect.sh L1 "" "step"')

## 5. Champion grid (Table C)

Best cell per **regime** (N bucket) per mode, per machine. Regimes: small N
(≤65K, cache-resident), mid (262K–1M, cache-spilling), large (≥4M, DRAM-bound).
No global winner — champions are regime + numbers + cell.

In [ ]:
def regime(n):
    if n <= 65_000: return 'small (<=65K)'
    if n <= 1_000_000: return 'mid (262K-1M)'
    return 'large (>=4M)'

if len(mins):
    mins2 = mins.assign(regime=mins['N'].map(regime))
    champs = (mins2.sort_values('ns_frame')
              .groupby(['machine_id','mode','regime'], as_index=False).first()
              [['machine_id','mode','regime','cell_short','N','ns_frame',
                'ns_particle','gbs_eff']])
    for (mid, mode), g in champs.groupby(['machine_id','mode']):
        print(f'\n=== champions: {mid} / {mode} ===')
        print(g[['regime','cell_short','N','ns_frame','ns_particle','gbs_eff']]
              .to_string(index=False))
else:
    print('no data — run scripts/collect.sh first')

## 6. Cross-machine comparison

Overlay the same cell across machines to see how the answer moves with
hardware. This is why `machine_id` is a dimension: a layout win on one
machine can be a wash on another (different cache sizes, different DRAM BW).

In [ ]:
if len(mins) and mins['machine_id'].nunique() > 1:
    for mode, g in mins.groupby('mode'):
        for cell, cg in g.groupby('cell_short', sort=False):
            fig, ax = plt.subplots(figsize=(7,4.5))
            for mid, mg in cg.groupby('machine_id'):
                mg = mg.sort_values('N')
                ax.plot(mg['N'], mg['ns_particle'], 'o-', label=mid, ms=4)
            ax.set_xscale('log'); ax.set_yscale('log')
            ax.set_xlabel('N'); ax.set_ylabel('ns / particle')
            ax.set_title(f'{cell} — {mode} (across machines)')
            ax.legend(fontsize=8, loc='best')
            fig.tight_layout(); plt.show()
elif len(mins):
    print(f'only one machine so far ({sorted(mins["machine_id"].unique())[0]}); '
          'collect on another to see the cross-machine view')

## 7. Hardware facts (per machine)

The anchors for every bandwidth/regime interpretation above.

In [ ]:
if len(hw):
    cols = [c for c in ['machine_id','cpu','physicalcpu','logicalcpu',
                        'l1dcachesize','l2cachesize','l3cachesize',
                        'cachelinesize','memsize_bytes','os','arch']
            if c in hw.columns]
    print(hw[cols].to_string(index=False))
else:
    print('no hardware sidecars found')